In [1]:
## 1.
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.figure(figsize=(10, 6))

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

In [3]:
## 2.
def preprocess(file_path='data/titanic.csv'):
    """빠른 데이터 전처리"""
    df = pd.read_csv(file_path)
    
    # 필수 전처리만 수행
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['Embarked'] = df['Embarked'].fillna('S')
    df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
    df['Fare'] = df['Fare'].fillna(df['Fare'].median())
    
    # 특성 선택
    features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
    X = df[features]
    y = df['Survived']
    
    return X, y

In [4]:
def evaluate_model(model, X_test, y_test, model_name="model"):
    # 예측
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1] # 분류모델은 확률 모델이다! 모든행, 두번째 컬럼

    # 기본 지표
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred)

    # 혼돈 행렬
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    # 결과 출력
    print(f"정확도: {accuracy}")
    print(f"정밀도: {precision}")
    print(f"재현율: {recall}")
    print(f"f1: {f1}")
    print(f"ROC_AUC: {roc_auc}")

    print(f"- TN: {tn}   FP: {fp}")
    print(f"- FN: {fn}   TP: {tp}")


    #classification_report(y_test, y_pred)

In [40]:
X, y = preprocess("data/titanic.csv")
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2,
                                                    stratify=y # (분류모델)불균형한 데이터를 우려하여 균질하게 섞어야 한다!
                                                    )

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
evaluate_model(rf, X_test, y_test, "Random Forest")

정확도: 0.8379888268156425
정밀도: 0.7857142857142857
재현율: 0.7971014492753623
f1: 0.7913669064748201
ROC_AUC: 0.830368906455863
- TN: 95   FP: 15
- FN: 14   TP: 55


In [ ]:
## 3
def plot_confusion_matrix(cm, model_name):
    """혼동 행렬 시각화"""
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['사망', '생존'], 
                yticklabels=['사망', '생존'],
                cbar_kws={'label': '예측 수'})
    plt.title(f'{model_name} - 혼동 행렬')
    plt.xlabel('예측')
    plt.ylabel('실제')
    plt.show()

def plot_metrics_comparison(results_dict):
    """여러 모델의 지표 비교"""
    metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
    models = list(results_dict.keys())
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    x = np.arange(len(metrics))
    width = 0.35
    
    for i, model in enumerate(models):
        values = [results_dict[model][metric] for metric in metrics]
        ax.bar(x + i*width, values, width, label=model, alpha=0.8)
    
    ax.set_xlabel('평가 지표')
    ax.set_ylabel('점수')
    ax.set_title('모델별 성능 비교')
    ax.set_xticks(x + width/2)
    ax.set_xticklabels(['정확도', '정밀도', '재현율', 'F1', 'ROC-AUC'])
    ax.legend()
    ax.set_ylim(0, 1)
    
    # 값 표시
    for i, model in enumerate(models):
        values = [results_dict[model][metric] for metric in metrics]
        for j, v in enumerate(values):
            ax.text(j + i*width, v + 0.01, f'{v:.3f}', 
                   ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.show()

Logistic Regression은 유일하게 Regression이 아니라 분류모델이다!

아이리스

In [41]:
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import StandardScaler
import numpy as np

iris = datasets.load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_scaled, y_train)

y_pred = clf.predict(X_test_scaled)

micro_f1 = f1_score(y_test, y_pred, average='micro')

print(f"Micro-F1 점수: {micro_f1:.4f}")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

macro_f1 = f1_score(y_test, y_pred, average='macro')
weighted_f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Micro-F1:    {micro_f1:.4f}")
print(f"Macro-F1:    {macro_f1:.4f}")
print(f"Weighted-F1: {weighted_f1:.4f}")

feature_importance = clf.feature_importances_
feature_names = iris.feature_names

for name, importance in zip(feature_names, feature_importance):
    print(f"{name}: {importance:.4f}")

Micro-F1 점수: 0.9000
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.82      0.90      0.86        10
   virginica       0.89      0.80      0.84        10

    accuracy                           0.90        30
   macro avg       0.90      0.90      0.90        30
weighted avg       0.90      0.90      0.90        30

Micro-F1:    0.9000
Macro-F1:    0.8997
Weighted-F1: 0.8997
sepal length (cm): 0.1163
sepal width (cm): 0.0150
petal length (cm): 0.4315
petal width (cm): 0.4372
